INSTAGRAM ENGAGEMENT ANALYSIS

EXECUTIVE SUMMARY

This project analyzes Instagram post engagement to identify content performance patterns.

Key Findings: • Photo posts slightly outperform video and carousel posts. • Moderate hashtag usage (2–3 hashtags) maintains stable engagement. • Emoji usage is consistent across all posts. • Posting time variation was limited in dataset.

Recommendations:

Prioritize high-quality photo content.
Maintain balanced content mix.
Use 2–5 targeted hashtags.
Continue emoji usage.
Implement structured weekly content calendar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
sns.set(style="whitegrid", palette="muted", font_scale=1.1)


In [ ]:
users=pd.read_csv(r"C:\Users\priya\Downloads\users.csv")
photos=pd.read_csv(r"C:\Users\priya\Downloads\photos.csv")
likes=pd.read_csv(r"C:\Users\priya\Downloads\likes.csv")
comments=pd.read_csv(r"C:\Users\priya\Downloads\comments.csv")
tags=pd.read_csv(r"C:\Users\priya\Downloads\tags.csv")
photo_tags=pd.read_csv(r"C:\Users\priya\Downloads\photo_tags.csv")
follows=pd.read_csv(r"C:\Users\priya\Downloads\follows.csv")

In [ ]:
users.head()
photos.head()
likes.head()
comments.head()

In [ ]:
users.shape
photos.shape
likes.shape
comments.shape

In [ ]:
print("Users:",users.shape)
print("Photos:",photos.shape)
print("Likes:",likes.shape)
print("Comments:",comments.shape)
print("Tags:",tags.shape)
print("Photo_Tags:",photo_tags.shape)
print("Follows:",follows.shape)


In [ ]:
follows.columns

In [ ]:
follows.columns = follows.columns.str.strip()
likes.columns = likes.columns.str.strip()
comments.columns = comments.columns.str.strip()
photos.columns = photos.columns.str.strip()
users.columns = users.columns.str.strip()
photo_tags.columns = photo_tags.columns.str.strip()
tags.columns = tags.columns.str.strip()

In [ ]:
follows.columns

In [ ]:
followers_count = follows.groupby('followee').size().reset_index(name='followers')

followers_count.head()

In [ ]:
likes.columns

In [ ]:
likes_count = likes.groupby('photo').size().reset_index(name='likes_count')

likes_count.head()

In [ ]:
comments.columns

In [ ]:
comments_count = comments.groupby('Photo id').size().reset_index(name='comments_count')

comments_count.head()

In [ ]:
photos.columns

In [ ]:
comments_count = comments.groupby('Photo id').size().reset_index(name='comments_count')

In [ ]:
photos_merged = photos.merge(
    likes_count,
    left_on='id',
    right_on='photo',
    how='left'
)

photos_merged = photos_merged.merge(
    comments_count,
    left_on='id',
    right_on='Photo id',
    how='left'
)

photos_merged = photos_merged.merge(
    followers_count,
    left_on='user ID',
    right_on='followee',
    how='left'
)

In [ ]:
photos_merged[['likes_count','comments_count','followers']] = \
photos_merged[['likes_count','comments_count','followers']].fillna(0)

In [ ]:
photos_merged['engagement_rate'] = (
    photos_merged['likes_count'] +
    photos_merged['comments_count']
) / photos_merged['followers']

photos_merged['engagement_rate'] = \
photos_merged['engagement_rate'].replace([np.inf, -np.inf], 0)

In [ ]:
photos_merged[['likes_count','comments_count','followers','engagement_rate']].describe()

In [ ]:
photos_merged['created dat'] = pd.to_datetime(
    photos_merged['created dat'],
    format='%d-%m-%Y %H:%M'
)

In [ ]:
photos_merged['hour'] = photos_merged['created dat'].dt.hour
photos_merged['day_name'] = photos_merged['created dat'].dt.day_name()

In [ ]:
hourly_engagement = photos_merged.groupby('hour')['engagement_rate'].mean().reset_index()

hourly_engagement.sort_values(by='engagement_rate', ascending=False).head()

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=hourly_engagement, x='hour', y='engagement_rate')
plt.title("Average Engagement by Posting Hour")
plt.show()

In [ ]:
photos_merged['hour'].value_counts()

In [ ]:
photo_type_engagement = photos_merged.groupby('photo type')['engagement_rate'].mean().reset_index()

photo_type_engagement.sort_values(by='engagement_rate', ascending=False)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=photo_type_engagement, x='photo type', y='engagement_rate')
plt.xticks(rotation=45)
plt.title("Engagement by Photo Type")
plt.show()

In [ ]:
comments[['Photo id','Hashtags used count']].head()

In [ ]:
hashtag_count = comments.groupby('Photo id')['Hashtags used count'].mean().reset_index()

photos_merged = photos_merged.merge(
    hashtag_count,
    left_on='id',
    right_on='Photo id',
    how='left'
)

photos_merged['Hashtags used count'] = \
photos_merged['Hashtags used count'].fillna(0)

In [ ]:
hashtag_engagement = photos_merged.groupby('Hashtags used count')['engagement_rate'].mean().reset_index()

hashtag_engagement.sort_values(by='engagement_rate', ascending=False)

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=hashtag_engagement, x='Hashtags used count', y='engagement_rate')
plt.title("Engagement vs Hashtag Count")
plt.show()

In [ ]:
photos_merged['hashtag_bucket'] = \
photos_merged['Hashtags used count'].round()

In [ ]:
hashtag_engagement_bucket = \
photos_merged.groupby('hashtag_bucket')['engagement_rate'].mean().reset_index()

hashtag_engagement_bucket.sort_values(by='engagement_rate', ascending=False)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=hashtag_engagement_bucket,
            x='hashtag_bucket',
            y='engagement_rate')
plt.title("Engagement by Hashtag Count")
plt.show()

In [ ]:
emoji_usage = comments.groupby('Photo id')['emoji used'].apply(
    lambda x: (x == 'yes').sum()
).reset_index(name='emoji_count')

In [ ]:
photos_merged = photos_merged.merge(
    emoji_usage,
    left_on='id',
    right_on='Photo id',
    how='left'
)

photos_merged['emoji_count'] = photos_merged['emoji_count'].fillna(0)

In [ ]:
photos_merged['emoji_bucket'] = photos_merged['emoji_count'].apply(
    lambda x: 'With Emoji' if x > 0 else 'No Emoji'
)

emoji_engagement = photos_merged.groupby('emoji_bucket')['engagement_rate'].mean().reset_index()

emoji_engagement

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(data=emoji_engagement,
            x='emoji_bucket',
            y='engagement_rate')
plt.title("Engagement: Emoji vs No Emoji")
plt.show()

In [ ]:
photos_merged['emoji_bucket'].value_counts()

STRATEGY RECOMMENDATIONS

Based on the engagement analysis of 257 posts, the following strategic insights are recommended as follows:

Prioritize high-quality photo posts, as they show slightly higher engagement compared to videos and carousels.
Maintain a balanced content mix, including educational videos and carousel guides.
Use 2–5 relevant and targeted hashtags to maintain stable engagement without overloading posts.
Continue emoji usage to maintain conversational tone and relatability.
Conduct A/B testing on posting time (8 AM vs evening slots) to identify optimal engagement windows.
These strategies aim to improve content consistency, audience interaction, and brand visibility.

CONCLUSION

This analysis examined content type performance, hashtag usage, emoji impact, and posting schedule patterns.

While the dataset showed limited variation in posting time and emoji usage, key engagement trends were identified. Photo-based content slightly outperformed other formats, and moderate hashtag usage maintained stable engagement levels.

Implementing a structured content calendar and controlled posting experiments can further enhance Instagram growth and audience engagement .